# Pelajaran 02 - Menjelajahi Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** adalah kerangka kerja terpadu untuk membangun agen AI. Ini menyediakan arsitektur yang bersih dan dapat dikomposisi dengan empat blok bangunan inti:

- **Client** – menghubungkan ke endpoint model AI dan menangani komunikasi
- **Agent** – membungkus client dengan instruksi dan definisi alat
- **Tools** – memperluas kemampuan agen dengan fungsi khusus yang dapat dipanggil model
- **Session** – menjaga riwayat percakapan untuk interaksi multi-giliran

Dalam pelajaran ini, kita akan membangun **agen pemesanan perjalanan** yang memeriksa ketersediaan tujuan menggunakan konsep-konsep ini.


## Pengaturan


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Memahami Arsitektur Kerangka Kerja Agen

Kerangka Kerja Agen Microsoft mengikuti arsitektur berlapis:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Sebuah `FoundryChatClient` terhubung ke deployment Azure OpenAI. Ini menangani autentikasi, format permintaan, dan parsing respons.
2. **Agent** – Dibuat dari client melalui `provider.create_agent()`, agen menggabungkan akses model dengan instruksi (prompt sistem) dan alat.
3. **Tools** – Fungsi Python yang dihias dengan `@tool` yang dapat dipanggil agen untuk melakukan tindakan atau mengambil data.
4. **Session** – Objek `AgentSession` (dibuat melalui `agent.create_session()`) yang menyimpan riwayat percakapan, memungkinkan dialog multi-putaran di mana agen mengingat konteks sebelumnya.

Mari kita bangun setiap lapisan langkah demi langkah.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Menambahkan Tools dengan Dekorator @tool

Tools memungkinkan agen melakukan tindakan lebih dari sekadar menghasilkan teks. Dekorator `@tool` mengubah fungsi Python biasa menjadi sesuatu yang dapat dipanggil oleh agen.

Poin penting:
- Gunakan `Annotated[type, "description"]` agar model memahami setiap parameter.
- Docstring menjadi deskripsi tool yang dilihat model.
- `approval_mode="never_require"` berarti tool berjalan otomatis tanpa konfirmasi pengguna.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Membuat Agen dengan Alat

Sekarang kita menggabungkan klien, instruksi, dan alat menjadi sebuah agen. `instructions` berperan sebagai prompt sistem — mereka mendefinisikan persona dan perilaku agen.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Percakapan Multi-Turn dengan Sesi

Sebuah `AgentSession` (dibuat melalui `agent.create_session()`) melacak semua pesan dalam sebuah percakapan. Dengan melewatkan sesi yang sama ke setiap panggilan `agent.run()`, agen memiliki akses ke seluruh riwayat percakapan dan dapat merujuk kembali ke pesan-pesan sebelumnya.

Kami melewatkan `tools=[check_destination_availability]` sehingga agen dapat memanggil pemeriksa ketersediaan kami selama setiap giliran.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Ringkasan

Dalam pelajaran ini Anda menjelajahi empat pilar dari Microsoft Agent Framework:

| Konsep | Apa yang Anda Pelajari |
|---------|------------------|
| **Klien** | `FoundryChatClient` menghubungkan ke Azure OpenAI dengan otentikasi berbasis kredensial |
| **Agen** | `provider.create_agent()` menggabungkan koneksi model dengan instruksi dan nama |
| **Alat** | Dekorator `@tool` membuka fungsi Python untuk dipanggil oleh agen |
| **Sesi** | `agent.create_session()` menjaga riwayat percakapan selama beberapa giliran |

Blok bangunan ini saling menyusun untuk menciptakan agen yang dapat melakukan percakapan alami, memanggil fungsi eksternal, dan mempertahankan konteks — fondasi untuk pola agentic yang lebih maju di pelajaran berikutnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
